# 01 - Flujo segmentacion_oraciones y ground truth oracional

Este notebook reemplaza los tres notebooks exploratorios anteriores en una vista unica: contrato del cierre, simulador offline, logs/feedback y evaluacion causal sobre la fuente real anotada.

La idea que queremos mostrar es simple: el cierre no corrige texto; solo decide cuando el buffer causal ya contiene una unidad semantica que se puede liberar al corrector.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

from segmentacion_oraciones.src.notebook_reports import card_grid, note_box
from segmentacion_oraciones.src.provider_factory import make_closure_provider
from segmentacion_oraciones.src.secuencias import load_sequence_ground_truth, evaluate_causal_sequence
from segmentacion_oraciones.src.simular_flujo import DEMO_TEXTS, run_simulation

pd.set_option('display.max_colwidth', 120)
OUT = Path('segmentacion_oraciones/outputs/notebooks/01_segmentacion_oraciones')
OUT.mkdir(parents=True, exist_ok=True)

## Contrato operativo

El sistema ve texto parcial, no video crudo. Cada vez que llega un clip/chunk nuevo, acumula texto desde el ultimo commit y toma una de tres acciones.

In [2]:
flow_html = """
<div style='display:flex;gap:8px;align-items:center;flex-wrap:wrap;margin:14px 0;font-family:Arial,sans-serif'>
  <div style='padding:10px 12px;border:1px solid #d8dee9;border-radius:8px;background:#f7f8fb'>RSV/VSR<br><b>chunks</b></div>
  <div style='font-size:22px;color:#596573'>?</div>
  <div style='padding:10px 12px;border:1px solid #d8dee9;border-radius:8px;background:#fff'>buffer textual<br><b>visible_context</b></div>
  <div style='font-size:22px;color:#596573'>?</div>
  <div style='padding:10px 12px;border:1px solid #2563eb;border-radius:8px;background:#eef5ff'>cierre<br><b>wait / commit / low_confidence</b></div>
  <div style='font-size:22px;color:#596573'>?</div>
  <div style='padding:10px 12px;border:1px solid #d8dee9;border-radius:8px;background:#fff'>corrector<br><b>solo si commit</b></div>
</div>
"""
display(HTML(flow_html))

## Fuente real anotada

Usamos `segmentacion_oraciones/ground_truth/charla_amor_desamor.json`. Tiene clips consecutivos y anotacion oracional con `start_clip`, `end_clip` y `commit_after_clip`.

In [3]:
sequence = load_sequence_ground_truth('segmentacion_oraciones/ground_truth/charla_amor_desamor.json')
clip_order = {clip.clip_id: clip.order for clip in sequence.clips}
span_lengths = [clip_order[s.end_clip] - clip_order[s.start_clip] + 1 for s in sequence.sentences]

display(card_grid([
    ('fuente', sequence.source_id[:34] + '...', None),
    ('clips', f'{len(sequence.clips):,}', 'unidades causales'),
    ('oraciones', f'{len(sequence.sentences):,}', 'ground truth oracional'),
    ('clips/oracion p50', f'{pd.Series(span_lengths).median():.0f}', 'span mediano'),
]))

display(pd.DataFrame([s.to_dict() for s in sequence.sentences[:8]])[['sentence_id','start_clip','end_clip','commit_after_clip','text']])

,sentence_id,start_clip,end_clip,commit_after_clip,text
0,s001,clip_0000,clip_0000,clip_0000,voy a antereccionar maria del cerro y ahora que hay un monton de gente en el chat les quiero decir algo.
1,s002,clip_0001,clip_0002,clip_0002,"la vida, aca voy a aparecer un hippie o esas finas que leen el horoscopo, pero la vida posta que se trata de energias."
2,s003,clip_0003,clip_0004,clip_0004,"entienden que estamos haciendo un stream a las 4 de la mañana en españa y tengo 24k de viewers hablando de futbol, a..."
3,s004,clip_0005,clip_0009,clip_0009,"estan saliendo todos los dias los rankings de twitch y a mi, yo ya tengo 10 11 años de streamer, no me interesa ranq..."
4,s005,clip_0013,clip_0015,clip_0015,"ojo, yo veo el numero y no les voy a negarme, me motivo, me motivo, las cosas estan saliendo bien y se los quiero ag..."
5,s006,clip_0016,clip_0018,clip_0018,"porque yo agradecido por mas que sea 2 3 4 20 50 500 22000, agraecido siempre, pero vale destacar que en un momento ..."
6,s007,clip_0019,clip_0019,clip_0019,"ojo, despues de que llueve sale el sol."
7,s008,clip_0020,clip_0021,clip_0021,"ahora solo me falta terminar el año ganando la velada, enamorandome y eventualmente el año que viene planeando tener..."


## Simulador offline

El simulador prueba el flujo completo sin GPU ni LLM externo: cierre heuristico + corrector identidad + logs/feedback. Sirve para validar contratos y fallbacks.

In [4]:
summary = run_simulation(
    DEMO_TEXTS,
    log_path=OUT / 'llm_logs.jsonl',
    feedback_path=OUT / 'feedback.jsonl',
)

display(card_grid([
    ('ejemplos', str(summary['count']), 'demo local'),
    ('commits', str(summary['actions']['commit']), 'pasan al corrector'),
    ('waits', str(summary['actions']['wait']), 'siguen acumulando'),
    ('fallbacks', str(summary['fallbacks']), 'deberia ser 0'),
]))
summary['latency_ms']

{'closure': {'p50': 0.137, 'p95': 0.2103},
 'correction': {'p50': 0.0456, 'p95': 0.049},
 'validation': {'p50': 0.0349, 'p95': 0.049},
 'logging': {'p50': 1.3757, 'p95': 2.6883}}

## Evaluacion causal de la heuristica

Esta es la baseline interpretable. Es util como piso y como feature, pero en real tiende a ser ansiosa: commitea temprano ante buffers largos.

In [5]:
heuristic = make_closure_provider('heuristic')
result = evaluate_causal_sequence(sequence, heuristic)

display(card_grid([
    ('expected commits', str(result['expected_commits']), None),
    ('predicted commits', str(result['predicted_commits']), None),
    ('precision commit', f"{result['commit_precision']:.3f}", None),
    ('recall commit', f"{result['commit_recall']:.3f}", None),
    ('early commits', str(result['early_commits']), 'error mas caro'),
    ('late waits', str(result['late_waits']), 'latencia'),
]))

errors = pd.DataFrame(result['rows'])
errors = errors[errors['expected_action'] != errors['predicted_action']].head(10)
display(errors[['clip_id','buffer_start_clip','expected_action','predicted_action','expected_sentence_id','reason','buffer_text']])

,clip_id,buffer_start_clip,expected_action,predicted_action,expected_sentence_id,reason,buffer_text
1,clip_0001,clip_0001,wait,commit,,contexto_suficiente_sin_conector_colgante,la vida aca voy a aparecer un hippie o esas finas que leen el horoscopo
3,clip_0003,clip_0003,wait,commit,,contexto_suficiente_sin_conector_colgante,entienden que estamos haciendo un stream a las 4 de la mañana en españa y tengo 24k de viewers hablando de futbol
5,clip_0005,clip_0005,wait,commit,,contexto_suficiente_sin_conector_colgante,yo ya tengo 10 11 años de streamer no me interesa ranquear pero te la blanqueo
6,clip_0008,clip_0008,wait,commit,,contexto_suficiente_sin_conector_colgante,ah buena poronga 23k ojo me la sube sabe por que me la sube
8,clip_0013,clip_0013,wait,commit,,contexto_suficiente_sin_conector_colgante,ojo yo veo el numero y no les voy a negarme me motivo me motivo
9,clip_0014,clip_0014,wait,commit,,contexto_suficiente_sin_conector_colgante,las cosas estan saliendo bien y se los quiero agradecer si yo tuviera 5k de viewers igual estaria motivado
10,clip_0015,clip_0015,commit,wait,s005,contexto_insuficiente,igual diria gracias por mirarme
11,clip_0016,clip_0015,wait,commit,,contexto_suficiente_sin_conector_colgante,igual diria gracias por mirarme porque yo agradecido por mas que sea 2 3 4 20 50 500 22000
12,clip_0017,clip_0017,wait,commit,,contexto_suficiente_sin_conector_colgante,agraecido siempre pero vale destacar que en un momento critico porque es un momento critico para mi
15,clip_0020,clip_0020,wait,commit,,contexto_suficiente_sin_conector_colgante,ahora solo me falta terminar el año ganando la velada enamorandome


## Lectura

El contrato ya esta estable y evaluable. El siguiente paso no es hacer mas heuristicas a mano, sino entrenar un cierre chico con datos ruidosos y medirlo contra esta baseline.

In [6]:
display(note_box('La heuristica queda como baseline/fallback y como feature. No es el modelo ganador esperado para produccion porque corta demasiado temprano cuando el buffer crece.', kind='warn'))